In [0]:
# ============================================================
# CELL 1 — Title & Setup
# ============================================================
# TAXI MEDALLION PIPELINE
# End-to-End Bronze → Silver → Gold
# All logic in utils/ — this notebook is orchestration only
# ============================================================

import sys
sys.path.append("/Workspace/Users/jaswanth.vudduru@gmail.com/taxi-medallion-pipeline")

from utils.bronze_ingestion import run_bronze
from utils.silver_cleaning  import run_silver, print_quality_report
from utils.gold_metrics     import compute_gold

print("✅ All modules imported successfully from utils/")

In [0]:
# ============================================================
# CELL 2 — Path Configuration
# ============================================================

from config import (
    RAW_DATA_PATH,
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    DEAD_LETTER_PATH,
    CHECKPOINT_PATH
)

# Map to variable names used in this notebook
SOURCE_PATH     = f"{RAW_DATA_PATH}/"
DEAD_LETTER     = DEAD_LETTER_PATH
QUARANTINE_PATH = f"{DEAD_LETTER_PATH}/quarantine"

print("📁 Paths configured from centralized config.py")

In [0]:
# ============================================================
# CELL 3 — BRONZE LAYER
# Raw ingestion with schema enforcement + dead-letter
# ============================================================

print("=" * 50)
print("BRONZE LAYER — Raw Ingestion")
print("=" * 50)

run_bronze(spark, SOURCE_PATH, BRONZE_PATH, DEAD_LETTER)

print("\n--- Bronze Table Preview ---")
display(spark.read.format("delta").load(BRONZE_PATH))

In [0]:
# ============================================================
# CELL 4 — SILVER LAYER
# Quality enforcement, type casting, deduplication
# ============================================================

print("=" * 50)
print("SILVER LAYER — Quality Enforcement")
print("=" * 50)

run_silver(spark, BRONZE_PATH, SILVER_PATH, QUARANTINE_PATH)

print("\n--- Silver Table Preview ---")
display(spark.read.format("delta").load(SILVER_PATH))

In [0]:
# ============================================================
# CELL 5 — QUALITY REPORT
# Programmatic validation checkpoint between layers
# This proves broken inputs never reach Gold
# ============================================================

print_quality_report(spark, BRONZE_PATH, SILVER_PATH, QUARANTINE_PATH)

print("\n--- Quarantined Rows (failed business rules) ---")
display(spark.read.format("delta").load(QUARANTINE_PATH))

In [0]:
# ============================================================
# CELL 6 — GOLD LAYER
# Business aggregates for reporting
# ============================================================

print("=" * 50)
print("GOLD LAYER — Business Metrics")
print("=" * 50)

m1, m2 = compute_gold(spark, SILVER_PATH, GOLD_PATH)

In [0]:
# ============================================================
# CELL 7 — VISUALIZATION 1
# Gold Metric 1: Revenue & Trip Summary per Vendor
# (Click chart icon → Bar chart → x: vendor_id, y: total_revenue)
# ============================================================

print("📊 Gold Metric 1 — Vendor Revenue Summary")
display(spark.read.format("delta").load(GOLD_PATH + "/vendor_revenue_summary"))

In [0]:
# ============================================================
# CELL 8 — VISUALIZATION 2
# Gold Metric 2: Payment Distribution
# (Click chart icon → Pie chart → label: payment_label, value: revenue_share_%)
# ============================================================

print("📊 Gold Metric 2 — Payment Type Distribution")
display(spark.read.format("delta").load(GOLD_PATH + "/payment_distribution"))

In [0]:
# ============================================================
# CELL 9 — DATA LIFECYCLE SUMMARY
# Show how data changed shape across all 3 layers
# ============================================================

bronze_df = spark.read.format("delta").load(BRONZE_PATH)
silver_df = spark.read.format("delta").load(SILVER_PATH)
gold1_df  = spark.read.format("delta").load(GOLD_PATH + "/vendor_revenue_summary")
gold2_df  = spark.read.format("delta").load(GOLD_PATH + "/payment_distribution")

print("""
╔══════════════════════════════════════════════════════════════╗
║              MEDALLION PIPELINE — DATA LIFECYCLE             ║
╠══════════════════════════════════════════════════════════════╣""")
print(f"║  BRONZE  │ {bronze_df.count():<6} rows │ {len(bronze_df.columns):<3} cols │ Raw, permissive, string types    ║")
print(f"║  SILVER  │ {silver_df.count():<6} rows │ {len(silver_df.columns):<3} cols │ Typed, deduped, rules enforced   ║")
print(f"║  GOLD 1  │ {gold1_df.count():<6} rows │ {len(gold1_df.columns):<3} cols │ Vendor revenue aggregates        ║")
print(f"║  GOLD 2  │ {gold2_df.count():<6} rows │ {len(gold2_df.columns):<3} cols │ Payment distribution metrics     ║")
print("╚══════════════════════════════════════════════════════════════╝")